# varn_lqg, step by step

This notebook explains how the improved controller `varn_lqg` (in `RamaCup/rama_controller/varn_lqg/`)
works, one step at a time. It uses the same AI4AO simulation as `SimpleKalman.ipynb`, and the same
functions as the competition controller, imported from its folder.

`varn_lqg` is a Kalman predictor on a VAR(N) model of all the modes, like `SimpleKalman`. It differs
in three ways:

1. **The order N is a choice** (`ORDER` below; `SimpleKalman` uses `AR = 3`).
2. **R = R_noise + R_aliasing**, from the WFS model, as in `SimpleKalman`. The reference
   controller `var_lqg` guesses it instead.
3. **The predictor is learnt on the telemetry.** The model-based predictor is the starting point.
   Its three matrices are then trained to minimise the residual they would leave. This is the new
   part (sections 8–11).

The pyramid adds a catch, covered in sections 3 and 9. It sees only a fraction `og` (the optical
gain) of the residual, and that fraction changes with the residual. Learning only for the residual
pushes the predictor to the edge of stability. The first version of this notebook did that, and the
learnt predictor diverged in closed loop. Two things fix it:
- **compensating the optical gain** in the pseudo open loop (section 3);
- **a robust step:** the learnt predictor is moved back towards the model-based one until it stays
  stable for an optical gain 30 % off (section 9).

The plan:

| section | what |
|---|---|
| 0 | setup: the simulation of `SimpleKalman` |
| 1 | `R_aliasing` |
| 2 | telemetry, with an integrator |
| 3 | the optical gain, and the pseudo open loop |
| 4 | `R = R_noise + R_aliasing` |
| 5 | why least squares is biased by the noise: a one-mode example |
| 6 | the VAR(N) model by least squares, and Q |
| 7 | the Kalman gain and the prediction |
| 8 | the controller, one frame |
| 9 | the learnt predictor, and its robust step |
| 10 | what the learning changed |
| 11 | the controllers in closed loop, on the same turbulence |
| 12 | starting from a wrong R |
| 13 | how this maps to `varn_lqg` in rama_cup |

The long parts are the simulations: sections 1, 2 and 11 run on the GPU. The fits and the training
(sections 6–10 and 12) run on the CPU and take a minute or two.

More background: `RamaCup/VARN_LQG_IMPROVEMENTS.md`.

## 0. Setup

The same simulation as `SimpleKalman.ipynb`. One addition: `varn_lqg`'s folder goes on the path,
so that its own functions are used.

- `varn_model.py`: the model-based part (least squares, Riccati, `A^delay`), pure numpy.
- `learned_predictor.py`: the training, with torch on the CPU.

The simulation runs with gradients off. The training turns them on only where it needs them.

In [ ]:
import copy
import sys
import time
from collections import deque
from pathlib import Path

import numpy as np
import pylab as plt
import torch
from tqdm import tqdm

from AI4AO import PyramidWFS, PhaseDataset, DeformableMirror, InfinitePhaseScreen
from AI4AO import imshow
from AI4AO.Utils import seed_everything
from kalman.Kalman_tools import estimate_R

# varn_lqg's own code: the functions the competition controller uses
VARN = Path.cwd().parent / "RamaCup" / "rama_controller" / "varn_lqg"
sys.path.insert(0, str(VARN))
from varn_model import fit_var, companion, kalman_gain, ahead
from learned_predictor import (train_predictor, robust_mix, hidden_loop_radius, residual_rms, held_out,
                               spectral_radius, filter_matrix)

device = "cuda"
torch.set_grad_enabled(False)      # the simulation needs no gradients; the training turns them on

In [ ]:
WFSParams = dict(
    {
        "Nres":30,
        "sampling":90/30,
        "D": 0.6,
        "centralObstruction": 0.,
        "useNoise": True,
        "Modulation": 2,
        "Wavelength": 635e-9,
        "Nphotons": [4.5, 6],  # Log range of number of photons in measurement
        "RON": [1, 3],  # Read-out noise in photons per pixel per frame
    }
)


## Atmosphere parameters
AtmosParams = dict(
    {
        "r0": [0.05, 0.05],  # Fired parameter range (m)
        "L0": [10, 30.0000],  # Outter scale range (m)
        "Nphases": 1,  # Number of phases in the batch
        "Layers": [5, 10],  # Number of layers in phase range
        "f_slope": 11.0 / 6.0,  # Slope of the spectrum (11/6) is the default
        "Scintillation": False # Use or not scinitillation
    }
)


## Loop parameters
LoopParams = dict(
    {
        "loopFrequency": 500,
        "delayFrames": 2,
        "windSpeedVector": [1, 10],
        "levelOfCorrection": [1., 1.],
        "loopGain": [0.2, 0.5],
        "loopLeak": [0.9,1.]
    }
)

## DM parameters
DMParams = dict(
    {
        "Nactuator": 11,
        "Nmodes": 80,
        "FlipLeftRight": False,
        "FlipTopBottom": False,
        "moffatParam": 2,
        "signedAmplitude": 1e-5,
        "MechCoupling": 0.35
    }
)

In [ ]:
dataset = InfinitePhaseScreen(WFSParams, AtmosParams, LoopParams, DMParams, device)
dataset.subharmonicLevels = 3

wfs = PyramidWFS(WFSParams, device)
wfs.eval()
dm = DeformableMirror(WFSParams, DMParams, device)
dm.eval()
M2C = dm.MakeZernikeM2C() * 10
M2C_T = M2C.T
Nmodes = M2C.shape[1]

modes = dm(M2C.T)
wfs.BuildReconstructionMatrix(modes)
z_inv = torch.linalg.pinv(modes.flatten(start_dim=-2))     # OPD -> modes: the true residual's modes (section 3)
print(f"{Nmodes} modes")

## 1. `R_aliasing`: what the DM cannot correct, seen as low-order modes

The DM corrects only `Nmodes` modes. The higher orders stay in the residual, and the pyramid folds
part of them back into the low-order modes it reconstructs: **aliasing**. For the Kalman filter, it
is extra noise on the measurement.

To measure it, reconstruct phases whose DM-correctable part has been removed
(`levelOfCorrection = [1, 1]`, as in `SimpleKalman`). What the WFS still reconstructs then is
aliasing alone.

One change from `SimpleKalman`: the WFS noise is **off** here. The noise is counted in `R_noise`
(section 4). With it on, it would be counted twice.

In [ ]:
AtmosParams_aliasing = copy.copy(AtmosParams)
AtmosParams_aliasing["Nphases"] = 16
dataset_aliasing = PhaseDataset(WFSParams, AtmosParams_aliasing, LoopParams, DMParams, device)
dataset_aliasing.generateClosedLoop = True

wfs.useNoise = False                  # aliasing only: the noise goes in R_noise
outputs_aliasing = []
for i in tqdm(range(1000)):
    batch = dataset_aliasing[0]
    wfs_frames = wfs(batch["opd"], batch["pupil"])
    outputs_aliasing.append(wfs.GetReconstructedOPD(wfs_frames))
wfs.useNoise = True

outputs_aliasing = torch.stack(outputs_aliasing).reshape(-1, Nmodes)
R_alias = torch.cov(outputs_aliasing.T)
print(f"R_alias: mean variance {R_alias.diagonal().mean().item():.3g}")

## 2. Telemetry, with an integrator

The predictor is learnt from **telemetry**: the WFS measurements `z_k` and the DM commands `u_k`.
Here they come from a run with an integrator, as in `SimpleKalman`.

In rama_cup, the optimizer gets the same telemetry from the server (`da.telemetry`), while
`varn_lqg` itself drives the loop.

Delay convention (as in `SimpleKalman`): the command `u_k` computed at frame `k` is first seen in
`z_{k+delay}`, with `delay = 2`.

Besides `z` and `u`, the loop also records the modes of the **true** residual OPD, the one the WFS
is about to measure (`residual_opd @ z_inv`). This is possible only in simulation, and section 3
uses it to measure the optical gain.

In [ ]:
delay = 2
SEED = 2028
seed_everything(SEED)
batch = dataset[0]
opd_gt, pupilGT = batch["opd"], batch["pupil"]
gain, leak = 0.3, 0.995
wfs.SetPhotonsAndRON(batch["nphotons"], batch["ron"])

z_estimated = torch.zeros(opd_gt.shape[0], Nmodes, device=device)
dm_buffer = deque([torch.zeros_like(z_estimated)] * delay, maxlen=delay)   # [u_{k-delay+1}, ..., u_k]
residual_opd = torch.zeros_like(opd_gt)
wfs_measurements, dm_commands, true_residuals = [], [], []

n_steps = 5000
for i in tqdm(range(n_steps)):
    if i > 0:
        batch = dataset[i]
        opd_gt, pupilGT = batch["opd"], batch["pupil"]

    true_residual = residual_opd.flatten() @ z_inv           # the modes the WFS is about to see
    wfs_frames = wfs(residual_opd, pupilGT)
    z_output = wfs.GetReconstructedOPD(wfs_frames)
    z_estimated = z_estimated * leak + gain * z_output       # the integrator

    dm_buffer.append(z_estimated)
    residual_opd = opd_gt - dm(dm_buffer[0] @ M2C_T)

    if i > n_steps * 0.1:                                    # skip the start of the loop
        wfs_measurements.append(z_output)
        dm_commands.append(z_estimated)
        true_residuals.append(true_residual)

# The fits run on the CPU, in numpy (float64): [frames, Nmodes]
z = torch.stack(wfs_measurements).reshape(-1, Nmodes).double().cpu().numpy()
u = torch.stack(dm_commands).reshape(-1, Nmodes).double().cpu().numpy()
true_res = torch.stack(true_residuals).reshape(-1, Nmodes).double().cpu().numpy()
print(f"telemetry: {z.shape[0]} frames of {Nmodes} modes")

## 3. The optical gain, and the pseudo open loop

The WFS measures the **residual**: the turbulence minus what the DM applied. Adding back the
command that was on the DM should give the turbulence alone, the **pseudo open loop**:

$$\mathrm{pol}_k = z_k + u_{k-\mathrm{delay}}$$

This is `estimate_atmospheric_state` in `kalman/`.

**The catch: the optical gain.** The reconstructor is calibrated with small pushes, where the
pyramid is linear. On a real residual it sees only a fraction `og` of it (per mode, below 1), and
less when the residual is large: $z_k \approx og \cdot (\phi_k - u_{k-\mathrm{delay}})$. Then

$$z_k + u_{k-\mathrm{delay}} = og\cdot\phi_k + (1 - og)\cdot u_{k-\mathrm{delay}}$$

This "pseudo open loop" still holds part of the controller's own commands. A predictor fed with it
closes a **hidden loop** on itself, with gain `(1 - og)`. A mild predictor survives that loop. An
aggressive one (the learnt predictor, section 9) can diverge.

**Compensation:** divide by the optical gain, $\mathrm{pol}_k = z_k / og + u_{k-\mathrm{delay}}$,
which gives $\phi_k$ again. In simulation `og` can be measured per mode: it is the slope of the
measurement `z` against the true residual's modes. On the bench there is no true residual. The
optical gain is then the `og` param of `varn_lqg` (set from the panel), or it is measured with a
dither, as in `DAOAdaptiveKalman.ipynb`.

The optical gain also depends on the controller: a better correction leaves a smaller residual,
and the pyramid then sees more of it. So `og` measured here, under the integrator, is lower than
under the predictors. Section 9 covers that error.

Sign: here the DM **subtracts** the command, so the command is the predicted turbulence itself. In
rama_cup the DM adds it, so `varn_lqg` outputs minus the prediction. The algebra is the same. In
rama_cup, the server provides `da.telemetry.pol` (with og = 1), and `varn_lqg` divides by its `og`.

In [ ]:
og = (z * true_res).sum(axis=0) / (true_res ** 2).sum(axis=0)     # per mode: z ~ og x true residual
print(f"optical gain under the integrator: median {np.median(og):.2f}, min {og.min():.2f}, max {og.max():.2f}")

pol = z[delay:] / og + u[:-delay]     # [frames, Nmodes]: the turbulence + the WFS noise (/ og)
print(f"pol: {pol.shape}, rms {pol.std():.3g}")

plt.figure(figsize=(8, 3))
plt.plot(og, ".-"); plt.axhline(1, color="gray", lw=0.5)
plt.xlabel("mode"); plt.ylabel("optical gain"); plt.title("what the WFS sees of the residual"); plt.show()

## 4. `R = R_noise + R_aliasing`

In the Kalman filter, `R` tells how noisy each measurement is: how much to trust the WFS against
the model's own prediction.

- **`R_noise`**: photon and read-out noise propagated through the reconstructor
  (`estimate_R`, as in `SimpleKalman`):
  $R_\mathrm{noise} = \frac{\mathrm{RON}^2}{N_\mathrm{ph}^2} M^T M + \frac{1}{N_\mathrm{ph}} (I_\mathrm{ref} M)^T M$,
  with $M$ the reconstruction matrix.
- **`R_aliasing`**: from section 1.

`var_lqg`, the reference controller, guesses `R = 0.3 / n_modes` instead. In `varn_lqg`, the two
terms are computed offline by `make_R.py` and loaded by the optimizer.

`R` is in the units of `z`. The pseudo open loop is `z / og`, so `R` is divided by
$og_i\,og_j$, as `varn_lqg`'s optimizer does with its `og`.

The plot compares, per mode:
- the measurement noise (`R`);
- the spread of the turbulence (`pol`);
- how much of `pol` the model cannot predict one frame ahead (`Q`, section 6).

In [ ]:
R_noise = estimate_R(wfs.reconstructionMatrix, wfs.reference_intensity, wfs.Nphotons, wfs.RON).squeeze()
R = (R_noise + R_alias).double().cpu().numpy() / np.outer(og, og)     # in the units of pol = z / og
R = 0.5 * (R + R.T)
# R must be positive definite (it is inverted in the Kalman gain): floor its eigenvalues, as the optimizer does
eigvals, eigvecs = np.linalg.eigh(R)
R = (eigvecs * np.clip(eigvals, 1e-6 * np.mean(np.diag(R)), None)) @ eigvecs.T

print(f"mean variance: R_noise {R_noise.diagonal().mean().item():.3g}, "
      f"R_alias {R_alias.diagonal().mean().item():.3g}, pol {pol.var(axis=0).mean():.3g}")

## 5. Why least squares is biased by the noise: a one-mode example

Least squares fits the model by predicting the next measurement from the previous ones. It treats
the previous measurements as exact, but they contain WFS noise.

Take one mode: AR(1) turbulence $\phi_{t+1} = a\,\phi_t + w_t$, measured as $p_t = \phi_t + v_t$.
Least squares finds

$$a_\mathrm{LS} = \frac{\mathrm{cov}(p_{t+1}, p_t)}{\mathrm{var}(p_t)} = a\,\frac{\sigma_\phi^2}{\sigma_\phi^2 + \sigma_v^2}$$

With the noise as large as the signal, it finds half of the true `a`. The model then thinks the
turbulence changes much faster than it does, and the predictor built on it is too timid.

The Kalman filter's `R` does not fix this. It sets how much the filter trusts each new measurement,
but the model `A` it trusts is already wrong. The learnt predictor (section 9) does not have this
bias.

In [ ]:
rng = np.random.default_rng(0)
a, T = 0.95, 100_000
phi = np.zeros(T)
for k in range(1, T):
    phi[k] = a * phi[k - 1] + rng.standard_normal()
noisy = phi + phi.std() * rng.standard_normal(T)            # noise as large as the signal

def least_squares(p):
    return np.dot(p[1:], p[:-1]) / np.dot(p[:-1], p[:-1])

print(f"true a {a}")
print(f"least squares, clean data  {least_squares(phi):.3f}")
print(f"least squares, noisy data  {least_squares(noisy):.3f}   (formula: {a / 2:.3f})")

## 6. The VAR(N) model by least squares, and Q

The model (`fit_var` in `varn_model.py`):

$$\mathrm{pol}_{t+1} = \mathrm{var}\,[\mathrm{pol}_t, \mathrm{pol}_{t-1}, \dots, \mathrm{pol}_{t-N+1}] + e_{t+1}, \qquad Q = \mathrm{cov}(e)$$

- `var` (`Nmodes × N·Nmodes`) is fitted by least squares, after removing the mean of `pol`.
- `Q` is the covariance of the one-step prediction residuals `e`.

As a state-space model (`companion`), the state is the last N frames,
$x_t = [\mathrm{pol}_t, \dots, \mathrm{pol}_{t-N+1}]$:

$$A = \begin{bmatrix} \mathrm{var} \\ I \quad 0 \end{bmatrix}, \qquad H = [I\ 0 \cdots 0]$$

Here, `A`'s first rows come from the model and the others shift the state by one frame. `Q`
lives in the newest block only. The older blocks are past frames, known exactly.

**The order N** is `ORDER`. A larger N describes the turbulence better but needs more frames
(`N·Nmodes` coefficients per mode). To try another one, change it and rerun from here.

In [ ]:
ORDER = 2                                    # N; the competition controller's --order
n = Nmodes

var, Q_top = fit_var(pol.T, ORDER)           # fit_var takes [n_modes, frames]
A = companion(var)
Q = np.zeros_like(A)
Q[:n, :n] = Q_top
print(f"VAR({ORDER}): var {var.shape}, A {A.shape}, largest |eigenvalue| of A {np.max(np.abs(np.linalg.eigvals(A))):.4f}")

plt.figure(figsize=(8, 4))
plt.semilogy(pol.std(axis=0), label="pol (turbulence + noise)")
plt.semilogy(np.sqrt(np.diag(Q_top)), label="not predicted one frame ahead (sqrt diag Q)")
plt.semilogy(np.sqrt(np.diag(R)), label="WFS noise + aliasing (sqrt diag R)")
plt.xlabel("mode"); plt.ylabel("rms"); plt.legend(); plt.show()

## 7. The Kalman gain and the prediction

From `A`, `Q` and `R`, the **steady-state Kalman gain** (`kalman_gain`) iterates the Riccati
equation until it converges:

$$K = P H^T (H P H^T + R)^{-1}, \qquad P \leftarrow A (P - K H P) A^T + Q$$

`P` and `K` do not depend on the measurements, so they are computed once, not every frame.
`SimpleKalman` recomputes them every frame, which made it 1.5 ms per frame.

The command must reach the WFS `delay` frames later. So it is the state predicted `delay` frames
ahead, keeping the current modes only:

$$C = (A^\mathrm{delay})_{[:n,\,:]}$$

`ahead` also handles fractional delays, e.g. 3.3 frames on the bench.

The three matrices `var`, `K`, `C` are the whole controller. They are the params `var`, `kalman`,
`predict` that `varn_lqg`'s optimizer sends to its controller.

In [ ]:
K, iterations = kalman_gain(A, Q, R, 2000)
C = ahead(A, delay)[:n]
model = (var, K, C)                          # the model-based predictor
print(f"K {K.shape} after {iterations} Riccati iterations; C {C.shape}")
print(f"filter F = (I - K H) A: largest |eigenvalue| {spectral_radius(var, K):.4f} (< 1: stable)")

## 8. The controller, one frame

Every frame, `varn_lqg`'s controller does three steps (`VarNLQG.update` in
`controller_varn_lqg.py`):

1. **pseudo open loop**: $\mathrm{pol}_k = z_k / og + u_{k-\mathrm{delay}}$ (section 3);
2. **Kalman filter**, with the constant gain: $x'_k = A\,x_{k-1}$, then
   $x_k = x'_k + K\,(\mathrm{pol}_k - H x'_k)$;
3. **command**: $u_k = C\,x_k$, the turbulence `delay` frames ahead.

That is three matrix-vector products. In the competition it takes about 20 µs per frame for
96 modes, against 2 ms per frame at 500 Hz.

Below, the same steps in torch for this simulation, and the integrator for comparison. Both take
`(z_k, u_{k-delay})` and return `u_k`.

If the output stops being finite, `varn_lqg`'s controller resets its state and outputs zero. Here
the closed loop (section 11) simply stops and reports the frame.

In [ ]:
class Predictor:
    # varn_lqg's controller with matrices (var, K, C), on the GPU. Here the DM subtracts the
    # command, so the command is +prediction (rama_cup: -prediction).
    def __init__(self, var, K, C, og=1.0):
        self.var, self.K, self.C = (torch.as_tensor(m, dtype=torch.float32, device=device) for m in (var, K, C))
        self.og = torch.as_tensor(og, dtype=torch.float32, device=device)
        self.n = self.var.shape[0]
        self.x = torch.zeros(self.var.shape[1], device=device)       # [pol_t, ..., pol_t-N+1], estimated

    def __call__(self, z, u_dm):
        pol = (z / self.og + u_dm).reshape(-1)                       # 1. pseudo open loop
        newest = self.var @ self.x                                   # 2. x' = A x: the newest block
        x_pred = torch.cat([newest, self.x[:-self.n]])               #    and the others shifted
        self.x = x_pred + self.K @ (pol - newest)                    #    x = x' + K (pol - H x')
        return (self.C @ self.x).reshape(1, -1)                      # 3. u = C x


class Integrator:
    def __init__(self, gain, leak):
        self.gain, self.leak, self.u = gain, leak, 0.0

    def __call__(self, z, u_dm):
        self.u = self.leak * self.u + self.gain * z
        return self.u

## 9. The learnt predictor, and its robust step

Sections 6–7 *derive* the predictor from a model: least squares gives `var`, `Q` is the residual
covariance, `R` is computed, and Riccati gives `K`. Each step has an error:
- `var` is biased by the noise (section 5);
- `Q` and `R` are estimates;
- `C = A^delay` compounds `A`'s error.

None of these steps looks at what the controller leaves behind.

The learnt predictor keeps the same three matrices and the same controller (section 8). It
**trains them directly** to minimise the residual the controller would leave on the telemetry:

$$p_k = C x_k, \qquad \mathrm{res}_t = \mathrm{pol}_t - p_{t-\mathrm{delay}}, \qquad
(\mathrm{var}, K, C) = \arg\min \sum_t \|\mathrm{res}_t\|^2$$

Here $x_k$ is the filter of section 8, run on `pol`. And $\mathrm{res}_t$ is what is left of the
turbulence at frame `t` after the command computed `delay` frames before, which is what the score
measures.

**Why the noise does not bias it.** The target $\mathrm{pol}_t = \phi_t + v_t$ is noisy, but $v_t$
is independent of the earlier frames the prediction uses. So
$E\|\mathrm{res}_t\|^2 = E\|\phi_t - p_{t-\mathrm{delay}}\|^2 + \sigma_v^2$. The noise adds a
constant that no matrix can change, so minimising the noisy residual also minimises the true one.
Least squares cannot do this: there the noise is in the inputs, which it treats as exact.

**How the training works** (`train_predictor` in `learned_predictor.py`):
- **Start:** the model-based predictor of section 7.
- **Optimiser:** Adam on `var`, `K`, `C`, with torch on the CPU.
- **Speed trick.** The filter is linear and time-invariant, so the prediction is a sum over the
  past frames, $p_k = \sum_j G_j\,\mathrm{pol}_{k-j}$, with $G_j = C F^j K$ and
  $F = (I - KH)A$. Each iteration computes $G_j$ (truncated when $F^j$ is negligible) and scores
  a random batch of 512 frames in one product. Running the filter frame by frame inside the
  gradient would be about 15 times slower.
- **Held-out check:** the last 30 % of `pol` is held out. The training stops when the residual
  there stops improving, and returns the best matrices on those frames, checked with the exact
  filter.
- **Safety:** an unstable filter is refused; the start is kept instead.

### The robust step

The training scores the predictor on recorded `pol`, as if `pol` did not depend on it. That holds
only if `og` is exactly right. If the WFS sees $\rho$ times what the controller assumes
($\rho = og_\mathrm{true} / og_\mathrm{used}$), part of the controller's own output comes back
through `pol` (section 3):

$$x_k = F x_{k-1} + K\,(1 - \rho)\,C\,x_{k-\mathrm{delay}}$$

`hidden_loop_radius` gives the spectral radius of this loop: below 1 it is stable. Minimising the
residual pushes the predictor towards the edge of stability, where a small error on `og` is
enough to diverge. The pyramid's `og` is never exact: it changes with the residual, and so with
the controller itself. Without this step, the learnt predictor of the first version of this
notebook diverged in about 20 frames.

So `robust_mix` takes the largest step from the starting predictor towards the learnt one,
$(1-\alpha)\,\mathrm{start} + \alpha\,\mathrm{learnt}$, whose hidden loop stays stable for $\rho$
from 0.7 to 1.3. `varn_lqg`'s optimizer does the same before it sends a learnt predictor
(config `robust_og`, `robust_max_radius`).

In [ ]:
t0 = time.time()
with torch.enable_grad():                   # the simulation turned gradients off
    *learnt_raw, info = train_predictor(pol, *model, delay, max_seconds=5, threads=torch.get_num_threads(), taps = 5)
learnt_raw = tuple(learnt_raw)
print(f"{info['iterations']} iterations, {time.time() - t0:.1f} s, {info['taps']} taps; "
      f"filter |eigenvalue| max {info['radius_F']:.3f}")

learnt, alpha, worst = robust_mix(model, learnt_raw, delay, rhos=(0.7, 1.0, 1.3))
print(f"robust step: alpha {alpha:.2f}, worst hidden-loop radius {worst:.3f}")
for name, m in (("model", model), ("learnt, no margin", learnt_raw), ("learnt, robust", learnt)):
    radii = [hidden_loop_radius(*m, delay, r) for r in (0.7, 1.0, 1.3)]
    print(f"  {name:18s} hidden-loop radius for og off by x0.7 / x1 / x1.3: " + " / ".join(f"{r:.3f}" for r in radii))

val = held_out(pol, delay)                  # the frames the training did not use
r_model, r_raw, r_learnt = (residual_rms(val, *m, delay) for m in (model, learnt_raw, learnt))
print(f"held-out residual rms (measured, includes the WFS noise): model {r_model:.4g}, "
      f"learnt no margin {r_raw:.4g} ({100 * (r_raw / r_model - 1):+.1f} %), "
      f"learnt robust {r_learnt:.4g} ({100 * (r_learnt / r_model - 1):+.1f} %)")

## 10. What the learning changed

The learnt matrices are no longer "physical". `var` is not the turbulence dynamics any more, and
`K` does not solve a Riccati equation. Only what they do together matters: the command is a linear
filter on the past measurements,

$$u_k = \sum_j G_j\,\mathrm{pol}_{k-j}, \qquad G_j = C F^j K$$

Many sets of matrices give the same filter. So compare the filters $G_j$, not the matrices:
- **left:** the overall size of $G_j$, i.e. how far back each predictor looks;
- **right:** the weight mode 0 gives to its own past values.

In [ ]:
def impulse_response(var, K, C, taps=40):
    F = filter_matrix(var, K)
    M, G = K.copy(), []
    for _ in range(taps):
        G.append(C @ M)
        M = F @ M
    return np.array(G)                      # [taps, n, n]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for name, m in (("model", model), ("learnt, no margin", learnt_raw), ("learnt, robust", learnt)):
    G = impulse_response(*m)
    axes[0].semilogy(np.linalg.norm(G, axis=(1, 2)), label=name)
    axes[1].plot(G[:, 0, 0], ".-", label=name)
axes[0].set(xlabel="frames in the past (j)", ylabel="||G_j||", title="how far back the predictor looks")
axes[1].set(xlabel="frames in the past (j)", ylabel="G_j[0, 0]", title="mode 0: weight on its own past")
for ax in axes:
    ax.legend()
plt.show()

In [ ]:
imshow(G)

## 11. The controllers in closed loop, on the same turbulence

All of them run on the **same** turbulence and the same WFS noise, starting from a flat DM:
- the integrator;
- the model-based predictor (sections 6–7);
- the learnt predictor without the robust step (section 9), to see what goes wrong;
- the learnt predictor with the robust step: what `varn_lqg` sends.

`InfinitePhaseScreen` repeats its turbulence when it is restarted (`dataset[0]`) with the same
seed. Each run restarts it with `EVAL_SEED`, which is different from the telemetry's, so the
controllers are scored on turbulence they were not fitted on. `SimpleKalman` compared the
integrator and the Kalman on different stretches of turbulence; here the comparison is like for
like.

The Strehl is the one of `SimpleKalman` (from the residual OPD variance). The table skips the first
200 frames, while the loops settle. A loop whose output stops being finite is stopped, and the
table gives the frame.

In [ ]:
n_eval = 2000
EVAL_SEED = SEED + 1


def run_loop(controller):
    seed_everything(EVAL_SEED)               # the same turbulence and WFS noise for every controller
    dm_buffer = deque([torch.zeros(1, Nmodes, device=device)] * delay, maxlen=delay)
    residual_opd = None
    strehl, psf_sum = [], 0.0
    for i in tqdm(range(n_eval)):
        batch = dataset[i]                   # i = 0 restarts the phase screen
        opd, pupil = batch["opd"], batch["pupil"]
        if residual_opd is None:
            residual_opd = torch.zeros_like(opd)
        wfs_frames = wfs(residual_opd, pupil)
        z_k = wfs.GetReconstructedOPD(wfs_frames)
        u_k = controller(z_k, dm_buffer[0])  # dm_buffer[0] = u_{k-delay}
        if not torch.all(torch.isfinite(u_k)):
            break                            # diverged
        dm_buffer.append(u_k)
        residual_opd = opd - dm(dm_buffer[0] @ M2C_T)
        strehl.append(100 * torch.exp(-(2 * torch.pi / dataset.wavelength * residual_opd).var(dim=(-2, -1))).item())
        psf_sum = psf_sum + wfs.GetPSF(residual_opd, pupil)
    return np.array(strehl), (psf_sum / max(len(strehl), 1)).squeeze().cpu()


controllers = {
    "integrator": Integrator(gain, leak),
    "varn_lqg, model": Predictor(*model, og=og),
    "learnt, no margin": Predictor(*learnt_raw, og=og),
    "learnt, robust": Predictor(*learnt, og=og),
}
results = {name: run_loop(ctrl) for name, ctrl in tqdm(controllers.items())}

plt.figure(figsize=(10, 4))
for name, (strehl, _) in results.items():
    plt.plot(strehl, label=name)
plt.xlabel("frame"); plt.ylabel("Strehl ratio (%)"); plt.legend(); plt.show()

for name, (strehl, _) in results.items():
    if len(strehl) < n_eval:
        print(f"{name:18s} diverged at frame {len(strehl)}")
    else:
        print(f"{name:18s} mean Strehl after frame 200: {strehl[200:].mean():6.2f} %")

In [ ]:
ok = [name for name, (strehl, _) in results.items() if len(strehl) == n_eval]
psfs = torch.stack([results[name][1] for name in ok])
print("mean PSFs, log scale:", " | ".join(ok))
imshow(torch.log(psfs))

## 12. Starting from a wrong R

The model-based gain is only as good as `R`. The learnt predictor uses `R` only through its
starting point. Here the training starts from predictors built with `R / 10` and `R × 10`.

In the synthetic simulation (`RamaCup/experiments/compare_learned.py`), the model-based predictor
was up to 35 % worse with a wrong `R`. The learnt predictor ended at almost the same place
whatever its start.

Here it is not always so. From a start far from good (`R × 10` in a test run), the training can
find nothing better on the held-out frames, and it keeps the start. A good `R` still matters,
through the starting point. In `varn_lqg` the training also starts from the predictor already in
place when that one is better, which avoids a bad start after the first fit.

This section is offline only, about two minutes on the CPU. It compares the held-out residuals
of the training alone, without the robust step.

In [ ]:
print(f"{'R used':10s} {'model':>8s} {'learnt':>8s}   held-out residual rms")
print(f"{'R':10s} {r_model:8.4g} {r_raw:8.4g}")
for label, scale in (("R / 10", 0.1), ("R x 10", 10.0)):
    K_s, _ = kalman_gain(A, Q, scale * R, 2000)
    start = (var, K_s, C)
    with torch.enable_grad():
        *learnt_s, _ = train_predictor(pol, *start, delay, max_seconds=60, threads=torch.get_num_threads())
    print(f"{label:10s} {residual_rms(val, *start, delay):8.4g} {residual_rms(val, *learnt_s, delay):8.4g}")

## 13. How this maps to `varn_lqg` in rama_cup

| this notebook | `varn_lqg` |
|---|---|
| 1, 4: `R_alias`, `R_noise` | `make_R.py`, once, offline: `R_noise.npy`, `R_aliasing.npy` in the folder |
| 2: integrator telemetry | `da.telemetry`, gathered while `varn_lqg` itself runs |
| 3: `og`, measured from the true residual | the `og` param (set from the panel), or a dither |
| 3: `pol = z / og + u_{k-delay}` | `da.telemetry.pol`, and `input / og - applied` in the controller |
| 6, 7: `fit_var`, `kalman_gain`, `ahead` | `optimizer_varn_lqg.py`, every 2000 frames, on up to the last 10 000 |
| 8: `Predictor` | `controller_varn_lqg.py`, every frame |
| 9: `train_predictor` | `optimizer_varn_lqg.py`, from the second fit on (config `learn`) |
| 9: `robust_mix` | `optimizer_varn_lqg.py`, before each learnt predictor is sent (`robust_og`, `robust_max_radius`) |

What the competition version adds:
- **It runs online.** The first fit sends the model-based predictor at once, so the loop corrects
  early. Later fits train from whichever is better, the new model or the predictor in place, and
  the training goes on from fit to fit.
- **It sends rarely.** A new predictor is sent only if it beats the one in place on the held-out
  frames by 0.5 %, since every send costs a few missed frames.
- **It handles the bench:** a fractional delay (3.3 frames), the optical gain `og` (the WFS sees
  `og × residual`), the DM limiter, and rama_cup's trial, which undoes a predictor that makes
  things worse.
- **Set `og` on the bench.** Without it (`og = 1`), the hidden loop of section 3 is strong. In
  this simulation, the robust step then had to keep the learnt predictor closer to the
  model-based one, and the gain was smaller.
- **Time limit:** each training stops after 15 s at most. The loop never waits for it.

What to try next:
- **Order:** change `ORDER` (section 6) to 3 or 4.
- **More telemetry:** a larger `n_steps` in section 2. More data help the learning most.
- **Fewer trained numbers:** `train_predictor(..., train_var=False)` trains only `K` and `C`.